In [1]:
# GPU check
import torch
print("GPU:", torch.cuda.is_available())

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install libraries
!pip install -q transformers==4.44.0 shap
print("Ready")

GPU: True
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Ready


In [2]:
!pip install -q shap
import shap
print("SHAP version:", shap.__version__)

SHAP version: 0.52.0


In [3]:
import torch
from transformers import RobertaForSequenceClassification, AutoTokenizer

DATA = '/content/drive/MyDrive/codebert_data'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RobertaForSequenceClassification.from_pretrained(f'{DATA}/codebert_final')
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
print("Model and tokenizer loaded from Drive")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Model and tokenizer loaded from Drive


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
import json

with open(f'{DATA}/test_tokenised.json') as f:
    test_data = json.load(f)

    # Find functions that are truly buggy (label=1) and predicted buggy
    examples = []
    for item in test_data:
        if item["label"] == 1:  # truly buggy
            input_ids = torch.tensor([item["input_ids"]]).to(device)
            mask = torch.tensor([item["attention_mask"]]).to(device)
            with torch.no_grad():
                out = model(input_ids=input_ids, attention_mask=mask)
                pred = torch.argmax(out.logits, dim=1).item()
                if pred == 1:  # correctly predicted buggy
                    examples.append(item)
                    if len(examples) >= 5:
                        break

    print(f"Found {len(examples)} correctly-flagged buggy functions to explain")

Found 5 correctly-flagged buggy functions to explain


In [8]:
import numpy as np
import scipy.special

def predict_proba(texts):
    results = []
    for text in texts:
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding="max_length")
        input_ids = enc["input_ids"].to(device)
        mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=mask)
            probs = scipy.special.softmax(out.logits.cpu().numpy(), axis=1)
            results.append(probs[0])
    return np.array(results)

print("Prediction function ready")

Prediction function ready


In [9]:
# Decode the first example's tokens back to readable code
example = examples[0]
code_text = tokenizer.decode(example["input_ids"], skip_special_tokens=True)
print("Example function (first 500 chars):")
print(code_text[:500])

Example function (first 500 chars):
static int xen_9pfs_connect(struct XenDevice *xendev)

{

    int i;

    Xen9pfsDev *xen_9pdev = container_of(xendev, Xen9pfsDev, xendev);

    V9fsState *s = &xen_9pdev->state;

    QemuOpts *fsdev;



    if (xenstore_read_fe_int(&xen_9pdev->xendev, "num-rings",

                             &xen_9pdev->num_rings) == -1 ||

        xen_9pdev->num_rings > MAX_RINGS || xen_9pdev->num_rings < 1) {

        return -1;

    }



    xen_9pdev->rings = g_malloc0(xen_9pdev->num_rings * sizeof(Xen9pf


In [10]:
# Build a SHAP explainer using a text masker
masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_proba, masker)

# Explain the first example (this takes a minute or two per function)
print("Generating SHAP explanation (this takes 1-3 minutes)...")
shap_values = explainer([code_text])
print("Explanation complete")

Generating SHAP explanation (this takes 1-3 minutes)...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:21, 21.37s/it]               

Explanation complete


In [11]:
# Text plot: highlights tokens by contribution
shap.plots.text(shap_values[0, :, 1])  # class 1 = buggy

In [12]:
html = shap.plots.text(shap_values[0, :, 1], display=False)
with open(f'{DATA}/shap_explanation_1.html', 'w') as f:
    f.write(html)
    print("Saved SHAP explanation to Drive: shap_explanation_1.html")

Saved SHAP explanation to Drive: shap_explanation_1.html
